# CaRL × Luna: inference and full search walkthrough

This walkthrough uses a **full held-out Sokoban board** and demonstrations from four disjoint training layouts. It shows the actual prompts and outputs of the **generator**, **goal-conditioned policy**, and **value** component, then traces the same controller rollout used in calibration and search.

**Start here:** select the project `.venv/bin/python` kernel and run from top to bottom. Defaults are `PROVIDER='mock'`, a real **12×12 / 4-box** trajectory-derived test board, and the fixed-horizon AdaSubS-style condition. Change the setup cell to use `PROVIDER='openai'` when you choose to run Luna; those calls use `.env` and incur API usage. `replay` only reads matching cached live responses. No keys are shown in the notebook.

The experiment uses **explicit board states and prompted Luna components**. CaRL's original generator, conditional policy and value use learned models. The full research proposal also has a learned latent encoder; it is not implemented by this explicit-state ICL walkthrough. The fixed and relative variants are research prototypes; no live performance is established by the offline demo.
**Campaign scope:** This independent walkthrough uses the earlier `tuned_fixed.json` settings and their legacy test split, not the frozen campaign policy, calibration, or untouched pilot cohort. For campaign validation use the configurations and commands in `analysis/luna_icl/campaign_100/campaign.md`.


**Latest measured generator:** open [luna_checkpoint_inspection.ipynb](luna_checkpoint_inspection.ipynb) for the saved 26-example checkpoint, full prompts and live validation results. Its runnable configuration is `configs/icl/experiments/luna_checkpoint26_retry2.json`; the saved complete search trace is in `analysis/luna_icl/campaign_100/generator_probe_v2/solver_validation/`. This walkthrough retains its separate offline mock profile.


## 1. Setup and experiment controls

`METHOD='fixed'` requests a target at constant trajectory offset **k**. Set `METHOD='relative'` to use policy-relative percentile **u**; that requires a completed calibration for the exact policy, provider and data. The rollout's `DEMO_CAP` is deliberately small and can be lower than the search/calibration cap. Change `FAMILY` or `BOARD_INDEX` to inspect other held-out boards. `generated_32x32x10` is a synthetic transfer distribution, not a trajectory from the 12×12 shard.

Prompts come from `BASE_CONFIG`, which inherits `configs/icl/experiments/base.json` → `prompts`. Change that JSON or override its nested fields in a child config. A policy-prompt change requires fresh calibration. Set `CACHE_DIR` to a new path to guarantee a cache miss for the demo. Set `SHOW_ALL_EXAMPLES = True` to display every actual few-shot message.

For the optional coordinate-augmented policy tested during live debugging, set `BASE_CONFIG = 'configs/icl/budget_sweeps/quick_coordinates_fixed.json'`. It adds coordinates derived from the input boards **only to the policy**, leaving the generator unchanged. Treat it as a pilot, not a validated default; relative search needs a new matching calibration. An existing `OPENAI_API_KEY` environment variable takes precedence over the key in `.env`.

The default is the trajectory-tuned pilot; run `python -m scripts.run_icl_search prepare-value` once to create its training-only value examples. The generator has no simulator verifier; value is returned by the model, including on terminal boards.
For a live run, set `CARL_CAMPAIGN_STAGE` before this notebook creates a client: the shared ledger uses `unassigned` with a zero spending limit by default. Existing environment variables take precedence over the repository `.env`; check the selected environment without displaying credentials.


In [ ]:
import json
import sys
from pathlib import Path
from typing import Final

# Usually automatic; set this only if the kernel starts outside the checkout/workspace.
REPO_PATH: str | None = None  # e.g. '/home/mtyrolski/phd/2026/CaRL'
anchors = (Path.cwd(), *Path.cwd().parents, *Path(sys.executable).parents)
candidates = (
    [Path(REPO_PATH).expanduser()]
    if REPO_PATH is not None
    else [candidate for anchor in anchors for candidate in (anchor, anchor / 'CaRL')]
)
ROOT = next((p.resolve() for p in candidates if (p / 'carl/icl').is_dir()), None)
if ROOT is None:
    raise RuntimeError(
        f'Cannot find the CaRL checkout from {Path.cwd()}. '
        'Set REPO_PATH above to the directory containing carl/icl, '
        'and select its .venv/bin/python as the notebook kernel.'
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'CaRL checkout: {ROOT} | Python: {sys.executable}')

# The checkout must be on sys.path before importing its modules.
# pylint: disable=wrong-import-position
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import plotly.graph_objects as go  # noqa: E402
from IPython.display import display  # noqa: E402

from carl.icl.boards import from_rows, level_key, tiles  # noqa: E402
from carl.icl.config import ExperimentConfig, Limits, load_config  # noqa: E402
from carl.icl.datasets import environment, load_data  # noqa: E402
from carl.icl.evaluation.board_graphics import board_comparison  # noqa: E402
from carl.icl.experiment import components_for_run, credentials  # noqa: E402
from carl.icl.search import search  # noqa: E402
from carl.icl.search_tree import SearchTreeRecorder, expansion_figure, tree_figure  # noqa: E402
from carl.icl.trace import trace_rollout  # noqa: E402
from carl.icl.types import Provider  # noqa: E402

# pylint: enable=wrong-import-position

# Change these in one place and re-run from this cell.
BASE_CONFIG = 'configs/icl/experiments/tuned_fixed.json'  # trajectory-tuned pilot
CACHE_DIR = 'analysis/luna_icl/cache'  # shared across experiments; change for fresh requests
SHOW_ALL_EXAMPLES = False
PROVIDER: Provider = 'mock'  # 'openai' costs money; 'replay' requires an exact live cache
FAMILY: Final = 'real_12x12x4'  # also 'generated_32x32x10' (synthetic)
BOARD_INDEX: Final = 0
METHOD: Final = 'fixed'  # 'fixed' (AdaSubS-style) or 'relative' (needs matching calibration)
SEED: Final = 11
HORIZON: Final = 4  # requested fixed k; training examples contain offsets 1..16
RELATIVE_SCALE: Final = 0.9  # policy-relative percentile; used only for METHOD='relative'
DEMO_CAP: Final = 4  # intentionally short; fixed uses min(DEMO_CAP, k), relative uses DEMO_CAP
ACTIONS: Final = ('UP', 'DOWN', 'LEFT', 'RIGHT')

if METHOD not in {'fixed', 'relative'}:
    raise ValueError("METHOD must be 'fixed' or 'relative'")
if BOARD_INDEX < 0 or not 2 <= HORIZON <= 16 or not 1 <= DEMO_CAP <= 16:
    raise ValueError('BOARD_INDEX must be >= 0; HORIZON 2..16; DEMO_CAP 1..16')
scale = float(HORIZON) if METHOD == 'fixed' else RELATIVE_SCALE
scales = [float(HORIZON), 1.0] if METHOD == 'fixed' else [RELATIVE_SCALE, 0.5 if RELATIVE_SCALE != 0.5 else 0.25]
base = load_config(BASE_CONFIG)
config = ExperimentConfig.model_validate({
    **base.model_dump(),
    'name': 'notebook_components', 'method': METHOD, 'scales': scales,
    'family': FAMILY, 'split': 'test', 'instances': BOARD_INDEX + 1, 'easy': False,
    'cache_dir': CACHE_DIR,
    'limits': Limits(max_requests=16, max_input_tokens=16000,
                     max_total_input_tokens=256000, max_output_tokens=16384,
                     wall_seconds=600, timeout_seconds=60).model_dump(),
})
print(f'Config: {BASE_CONFIG}; prompts: configs/icl/experiments/base.json (or overrides)')
print(f'Few-shot examples: {config.few_shot}; cache: {config.path("cache_dir")}')
print(f'Coordinate features for policy: {config.policy_coordinate_features}')
print(f'Provider: {PROVIDER} | model: {config.model} | method: {METHOD} | scale: {scale:g}')
print(f'Local logical budget: {config.limits.max_requests} requests; live calls begin only in section 5.')

## 2. Load a held-out board and inspect provenance

The prepared training and test data were exported from the trajectory shard. `load_data` checks provenance, board validity and disjoint wall/goal layouts across training, validation and test. The original test solution is not passed to any inference component. If data is missing, run `make icl-prepare` from CaRL first.

In [ ]:
try:
    training, instances = load_data(config)
except FileNotFoundError as exc:
    raise FileNotFoundError('Prepared data missing: run make icl-prepare in CaRL.') from exc
instance = instances[BOARD_INDEX]
initial = from_rows(instance['state'])
env = environment(initial)
box_count = int(np.isin(tiles(initial), [3, 4]).sum())
print(f"{instance['id']}: {initial.shape[0]}×{initial.shape[1]}, {box_count} boxes, full initial state")
print('Prepared test file:', config.path('data_dir') / FAMILY / 'test.json')
print('Few-shot data:', config.path('data_dir') / 'training.json')
print('Source shard:', config.path('trajectory_source'))
print('Distinct test layout:', all(level_key(initial) != level_key(from_rows(c['state'])) for c in training['contexts']))
board_comparison(initial, initial, ('Held-out state', 'Held-out state')).show()

### Board legend

Dark blue: wall · cream: floor · green ring: goal · orange crossed square: box · blue circle: player. A box or player on a goal retains the goal marking. Graphics follow the thesis figures and work for both board sizes.

## 3. Construct and inspect the actual few-shot inputs

This cell does not call the model. The generator receives verified state → future-state examples with the requested fixed `k`; the policy receives state, future target and the **next action**, using trajectory offsets 1, 4, 8 and 16; the tuned value receives four states from each training trajectory with their remaining action counts; the returned count is converted to `γ ** remaining_actions`. These are in-context analogues of CaRL's components, not trained checkpoints. Retrieval here is direct, so no embeddings call is made.

For the relative variant, matching completed calibration must exist before this cell: the frozen policy is rolled out against reference targets; success weights and first-hit times produce `u`. Calibration of real Luna is a separate paid experiment and is not started by this notebook.

In [ ]:
components = components_for_run(config, PROVIDER, training, initial)
api = components.api
examples = components.generator_examples(initial, scale, resolve_embeddings=False)
example = examples[0]
demo = from_rows(example['state'])
demo_target = from_rows(example['subgoal'])
summary = []
for index, item in enumerate(examples):
    if METHOD == 'fixed':
        label = item['k']
    else:
        if 'u' not in item:
            raise ValueError(f"Missing percentile for calibrated context {item['context']}")
        label = item['u']
    source = from_rows(item['state'])
    example_env = environment(source)
    replayed = source.copy()
    for action in item['actions']:
        replayed = np.asarray(example_env.next_state(replayed, action))
    matches = len(item['actions']) == item['k'] and np.array_equal(replayed, from_rows(item['subgoal']))
    if not matches:
        raise ValueError(f"Training example {item['context']} does not replay to its target")
    summary.append({'example': index + 1, 'training_context': item['context'],
                    'trajectory_offset_k': item['k'], 'generator_label': label,
                    'replay_verified': matches})
display(pd.DataFrame(summary))
board_comparison(demo, demo_target, ('Selected generator example: state', 'Its demonstrated future state')).show()
policy_examples = components.policy_examples()
print('Policy example first-action labels:', [ACTIONS[int(np.argmax(answer['probabilities']))] for _, answer in policy_examples])

value_preview = components.value_payload(initial)
value_answers = [json.loads(message['content'][0]['text']) for message in value_preview['input'] if message['role'] == 'assistant']
print(f'Actual value examples: {len(value_answers)}; answer labels:', value_answers)

### Optional: inspect the exact request messages

The following cell builds requests locally and sends nothing. The generator's query contains only the current board, task goals, scale and requested count. The tuned value also receives board-derived coordinate and distance features. One actual example and the query are displayed here; the policy request is shown after a target is generated. The tuned profile includes four generator/policy examples and sixteen value examples (four states from each of four training trajectories).

In [ ]:


def inspect_request(name: str, request: dict) -> None:
    """Show one actual few-shot exchange and the current query, without sending it."""
    messages = request['input']
    print(f"{name}: {len(messages)} messages, {sum(m['role'] == 'assistant' for m in messages)} examples")
    print('Output token limit:', request['max_output_tokens'], '| format:', request['text']['format']['type'])
    display({'developer_instructions': messages[0]['content'][0]['text'],
             'first_example_input': json.loads(messages[1]['content'][0]['text']),
             'first_example_answer': json.loads(messages[2]['content'][0]['text']),
             'query': json.loads(messages[-1]['content'][0]['text'])})
    if SHOW_ALL_EXAMPLES:
        display({'all_messages': messages})

generator_request = components.generator_payload(initial, scale, resolve_embeddings=False)
value_request = components.value_payload(initial)
inspect_request('generator', generator_request)
inspect_request('value', value_request)
# Full request dictionaries include all examples: display(generator_request), display(value_request)
# Never paste API keys into notebook cells or outputs.

## 4. Value and generator

The generator proposes an explicit board. Validation rejects malformed layouts, changed walls/goals/box counts, duplicates and self-copies. A structurally valid board **is not guaranteed reachable**. The value is a model estimate of discounted continuation return, not a calibrated success probability. Neither a heuristic mock output nor a cached replay measures current Luna quality.

The offline mock generator picks a heuristic one-step successor and ignores `k`/`u`; its reachability and solve outcomes cannot test temporal-scale learning. Use real Luna runs for that question.

## 5. Run inference: value and a subgoal

**First live-call boundary.** With `PROVIDER='openai'`, this cell can send Responses requests and loads credentials from `.env`; with `mock`, it remains offline. Model output and retries are bounded by the config; a cache hit still consumes logical budget. An invalid or empty proposal is shown as such and is never replaced by a simulator-generated target. A valid-looking generated board can still be unreachable; the subsequent policy rollout provides the observed result.

In [ ]:
credentials(PROVIDER)
usage_before = dict(api.metrics)
initial_value = components.value(initial)
proposals = components.propose(initial, scale)
target = proposals[0] if proposals else None
print(f'Initial value: {initial_value:.4f} | valid proposals: {len(proposals)}')
print('Generator diagnostics:', components.diagnostics)
print('Provider attempts (mock = simulated):', api.metrics['requests'] - usage_before['requests'],
      '| local cache hits:', api.metrics['cache_hits'] - usage_before['cache_hits'])
if target is not None:
    board_comparison(initial, target, ('Current state', f'Generated target ({METHOD}={scale:g})')).show()
    policy_request = components.policy_payload(initial, target)
    inspect_request('policy', policy_request)
else:
    print('No valid target: the controller section will be skipped.')

## 6. Policy: sample and execute real simulator actions

`trace_rollout` wraps the **same `search.rollout`** used by calibration and the full search: it samples the four probabilities in `[UP, DOWN, LEFT, RIGHT]` order, executes one environment action at a time, counts no-ops, and stops at the exact requested target, a solved task or the cap. The trace records each distribution and transition without calling the policy twice. `SEED` fixes local categorical sampling, not model generation.

The notebook uses `min(DEMO_CAP, k)` for fixed horizon and `DEMO_CAP` for relative. This is a short demonstration; the full fixed-horizon experiment normally uses `k`, and the full relative experiment uses its separately configured cap `L`.

In [ ]:
attempt = None
history = []
if target is not None:
    execution_cap = min(DEMO_CAP, HORIZON) if METHOD == 'fixed' else DEMO_CAP
    attempt, steps = trace_rollout(env, components, initial, target,
                                   cap=execution_cap, rng=np.random.default_rng(SEED))
    for index, step in enumerate(steps, start=1):
        history.append({'step': index, 'action': ACTIONS[step.action], 'changed': step.changed,
                        **{name: float(prob) for name, prob in zip(ACTIONS, step.probabilities)}})
        print(f'Step {index}: {ACTIONS[step.action]} | changed={step.changed}')
        go.Figure(go.Bar(x=list(ACTIONS), y=step.probabilities)).update_layout(
            title=f'{PROVIDER} policy at step {index}', yaxis={'range': [0, 1], 'title': 'Probability'},
            template='plotly_white', height=250,
        ).show()
        board_comparison(step.state, step.successor, ('Before', f'After {ACTIONS[step.action]}')).show()
    print(f'Exact target reached: {attempt.reached} | task solved: {attempt.solved} | attempted actions: {len(attempt.actions)}')
    display(pd.DataFrame(history))

## 7. Value at the endpoint and usage

The full search asks value to order frontier states at a fixed scale; it does **not** evaluate every intermediate action. The endpoint value below is an additional diagnostic so you can inspect the component. `logical_*` counters include cache hits; provider attempts exclude cache hits; for mock they are simulated local attempts, not network requests. Provider-reported token usage excludes local hits. Failed requests can still consume resources.

In [ ]:
if attempt is not None:
    endpoint_value = components.value(attempt.state)
    print(f'Initial value {initial_value:.4f} → endpoint value {endpoint_value:.4f} (diagnostic, not ground truth)')
    board_comparison(initial, attempt.state, ('Start', 'Actual endpoint')).show()
display(pd.DataFrame([dict(api.metrics)]))

## 8. Run the complete bounded solver

This is the same `search(...)` used by the experiments: it evaluates the initial board, orders the frontier by scale and value, proposes **explicit boards**, rolls out the policy and falls back to primitive actions. The selected board is the original held-out **full** puzzle, not its one-move fixture. The seed, action budget and expansion budget are below; termination without a solution is a valid result.

The mock provider runs automatically and makes **no network calls**. For Luna, sections 5–7 already make component calls; the additional solver run can consume up to `SOLVER_REQUEST_BUDGET` logical requests, so set `RUN_SEARCH = True` only when ready. Requests include cached requests; inspect the usage table for actual provider attempts. A fresh client and independent counters are used for each run.

The recorder observes the existing solver and performs no model calls. Its nodes are **attempted paths**, including no-ops, duplicates and failed subgoals; these leaves were not necessarily inserted into the frontier. A board can occur in multiple nodes and one path can be expanded at several scales. This is the explicit-state ICL solver's trace, not the old trained CaRL solver's internal `SearchTreeNode`.
The production queue prioritizes larger scale first and value second. Scale zero triggers primitive fallback without calling the generator. Every attempted simulator action counts, including no-ops and rejected subgoal attempts; remaining budget truncates each rollout and each fallback loop. Task completion is checked after every action, including before an exact subgoal hit.

A goal-only direct controller is not available in this setup: the current policy requires a complete target board, and the existing direct mode constructs a solved board. Do not interpret that mode as goal-only execution.


In [ ]:
RUN_SEARCH = False  # opt in to the additional full search when PROVIDER is openai/replay
RUN_MOCK_SEARCH = True  # the full held-out board runs automatically offline
SOLVER_ACTION_BUDGET = 64
SOLVER_EXPANSION_BUDGET = 64
SOLVER_REQUEST_BUDGET = 256

search_config = ExperimentConfig.model_validate({
    **config.model_dump(),
    'action_budget': SOLVER_ACTION_BUDGET,
    'max_expansions': SOLVER_EXPANSION_BUDGET,
    'limits': Limits(max_requests=SOLVER_REQUEST_BUDGET, max_input_tokens=16000,
                     max_total_input_tokens=4_096_000, max_output_tokens=131_072,
                     wall_seconds=600, timeout_seconds=60).model_dump(),
})

def run_traced_search(board, run_config):
    """Run the production search with an observation-only tree and fresh client."""
    credentials(PROVIDER)
    run_components = components_for_run(run_config, PROVIDER, training, board)
    recorder = SearchTreeRecorder()
    result = search(run_components.env, run_components, board, run_config, SEED, observer=recorder)
    assert 0 <= result.attempted_actions <= run_config.action_budget
    assert len(recorder.expansions) == result.expansions
    assert sum(len(node.actions) for node in recorder.nodes) == result.attempted_actions
    return result, recorder, run_components

outcome = tree = search_components = None
if RUN_SEARCH or (PROVIDER == 'mock' and RUN_MOCK_SEARCH):
    outcome, tree, search_components = run_traced_search(initial, search_config)
    display(pd.DataFrame([tree.summary()]).T.rename(columns={0: 'full held-out board'}))
    print('Model diagnostics:', search_components.diagnostics)
    display(pd.DataFrame([dict(search_components.api.metrics)]))
else:
    print('Full search skipped. Set RUN_SEARCH = True to run it.')

## 9. Search tree and frontier accounting

Node labels are IDs for the inspection cell below. Hover a node for its actual segment and status; green edges mark the exact solution ancestry when one exists. The progress chart samples actions **before** each expansion and queue length **after** the pop; queue entries include other scales and potentially stale entries, so it is not a count of unique states or the maximum queue size. `accepted_subgoals` means exact target hits, even if the endpoint was a duplicate.

In [ ]:
if tree is not None:
    tree_figure(tree).show()
    expansion_figure(tree).show()
    display(pd.DataFrame(tree.rows()).head(40))
    if len(tree.nodes) > 40:
        print(f'Showing 40 of {len(tree.nodes)} attempt nodes; use tree.rows() for all.')

### Inspect a node

Set `NODE_ID` to a label in the tree and rerun this cell. The right panel is the **actual endpoint**, including for failed attempts. For a generated subgoal, a second figure compares the proposed board to that endpoint; a structurally valid target need not be reachable.

In [ ]:
NODE_ID = 0
if tree is not None:
    if not 0 <= NODE_ID < len(tree.nodes):
        raise ValueError(f'NODE_ID must be between 0 and {len(tree.nodes) - 1}')
    node = tree.nodes[NODE_ID]
    parent = tree.nodes[node.parent_id] if node.parent_id is not None else node
    display(pd.DataFrame([tree.rows()[NODE_ID]]))
    board_comparison(from_rows(parent.state), from_rows(node.state),
                     (f'Parent {parent.node_id}', f'Actual endpoint {node.node_id}')).show()
    if node.proposal is not None:
        board_comparison(from_rows(node.proposal), from_rows(node.state),
                         ('Proposed exact board', 'Actual endpoint')).show()

## 10. Verify and replay the solution

A success is shown only after replaying **every returned primitive action** from the starting board in the real Sokoban environment and checking `is_solved`. Unsuccessful search does not produce a solution path. To inspect a move, change `REPLAY_STEP` and rerun the final cell; both boards use the thesis-style box graphics.

In [ ]:
def verified_replay(board, result):
    """Replay the solver's returned actions in the actual environment."""
    if not result.solved:
        return []
    replay_env = environment(board)
    states = [board.copy()]
    for replay_action in result.action_path:
        states.append(np.asarray(replay_env.next_state(states[-1], replay_action)))
    if not replay_env.is_solved(states[-1]):
        raise AssertionError('Search reported success but its action path did not solve the board')
    return states

replay_states = verified_replay(initial, outcome) if outcome is not None else []
if outcome is not None and replay_states:
    print(f'Verified solution: {len(replay_states) - 1} simulator actions')
    display(pd.DataFrame({'step': range(1, len(replay_states)),
                          'action': [ACTIONS[a] for a in outcome.action_path]}))
else:
    print('No solution to replay for this run.')

In [ ]:
REPLAY_STEP = 0  # between 0 and the number of solution actions
if replay_states:
    if not 0 <= REPLAY_STEP < len(replay_states):
        raise ValueError(f'REPLAY_STEP must be between 0 and {len(replay_states) - 1}')
    next_step = min(REPLAY_STEP + 1, len(replay_states) - 1)
    board_comparison(replay_states[REPLAY_STEP], replay_states[next_step],
                     (f'Step {REPLAY_STEP}', f'Step {next_step}')).show()

## 11. Offline solved-path smoke check

The full puzzle may exhaust its budget. To demonstrate a successful search and replay independently, the mock run below uses the **same selected held-out layout** at its prepared one-move-from-solved fixture. This is a wiring check, **not** a solve-rate result for the original puzzle. It has its own client, counters, tree and budgets. It never runs automatically with Luna.

In [ ]:
RUN_MOCK_FIXTURE = PROVIDER == 'mock'
fixture_outcome = fixture_tree = None
if RUN_MOCK_FIXTURE:
    fixture_config = ExperimentConfig.model_validate({**search_config.model_dump(), 'easy': True})
    _, fixture_instances = load_data(fixture_config)
    fixture_board = from_rows(fixture_instances[BOARD_INDEX]['state'])
    fixture_outcome, fixture_tree, fixture_components = run_traced_search(fixture_board, fixture_config)
    display(pd.DataFrame([fixture_tree.summary()]).T.rename(columns={0: 'one-move fixture'}))
    tree_figure(fixture_tree).show()
    fixture_replay = verified_replay(fixture_board, fixture_outcome)
    if fixture_replay:
        print('Verified fixture actions:', [ACTIONS[a] for a in fixture_outcome.action_path])
        board_comparison(fixture_replay[0], fixture_replay[-1], ('Fixture start', 'Verified solved')).show()
    else:
        print('The bounded mock search did not solve the fixture; inspect its tree above.')

## Next steps

Use `configs/icl/experiments/` and `scripts/run_icl_search.py` for calibrated matched experiments. Open **luna_icl_evaluation.ipynb** to inspect saved outcomes and solve-rate-versus-budget plots. Prompts and few-shot settings are in `configs/icl/experiments/base.json` and its overrides. See `analysis/luna_icl/PROPOSAL_ALIGNMENT.md` for the explicit-state scope. Mock outputs test wiring only; they do not measure Luna.